# Workshop 1 · Hunt for drug, target and disease data

**AI- and data-driven drug design – An introduction**

Today you will learn to collect drug, target and disease data **with code**, so that a search you would otherwise click through by hand can be repeated for any drug, target or disease, and rerun when the databases are updated. At the end (**Part D**) you will apply exactly the same steps to a disease of your choice, possibly from your own research, so keep that in mind as you go.

**You do not need to know Python.** All the code is written for you. Your job is to run it, read the results, and change a few marked words.

### How to use this notebook
* A notebook is made of **cells**. Grey boxes are **code cells**; the rest is text.
* To run a code cell, click the **▶ play button** that appears on its left when you hover over it (or click in the cell and press **Shift + Enter**). The result appears just below the cell.
* Run the cells **in order, from top to bottom**. Later cells use results from earlier ones; if something fails, check that you ran the cells above it.
* Some cells ask you to change a word, for example a drug name. These are marked with ✏️ and the text above them says exactly what to change. The first one is in **A1**.
* A small number in square brackets next to a cell, like [3], means it has run. If you see an error, read its last line, check what you changed, and ask an instructor.

### Colour code in the text
* 🔎 <font color="#1565C0">**What this query does:**</font> explains what the code cell below asks the database.
* 📊 <font color="#2E7D32">**What you get:**</font> explains what the output shows and how to read it.
* ▶ <font color="#C62828">**Your turn:**</font> tells you exactly what to do.

### Contents
| Part | What you do
|---|---|
| **Warm-up** | Look up a drug, a target and a disease on the websites (no code) |
| **0 · Setup** | Prepare the notebook |
| **A · Open Targets** | Follow one thread: drug **semaglutide** → its target **GLP1R** → the disease **type 2 diabetes** |
| **B · ChEMBL** | Continue the thread: an approved diabetes drug, **sitagliptin** → its target **DPP4** → potent compounds → similar molecules |
| **C · Worked example** | Find new targets for type 2 diabetes, step by step |
| **D · Your project** | Repeat Part C for your own disease |

The parts form **one continuous story**: each step uses a result from the step before. The arrows (→) in the headings show where each input comes from.

### Where to find help and documentation
* **Open Targets:** [GraphQL API documentation](https://platform-docs.opentargets.org/data-access/graphql-api) · [GraphQL browser to try queries](https://api.platform.opentargets.org/api/v4/graphql/browser) · [Community forum](https://community.opentargets.org/)
* **ChEMBL:** [API documentation and list of resources](https://www.ebi.ac.uk/chembl/api/data/docs) · [Python client with examples](https://github.com/chembl/chembl_webresource_client) · [ChEMBL web interface](https://www.ebi.ac.uk/chembl/)
* **Python tools used here:** [pandas (tables)](https://pandas.pydata.org/docs/user_guide/10min.html) · [RDKit (drawing molecules)](https://www.rdkit.org/docs/GettingStartedInPython.html) · [Google Colab introduction](https://colab.research.google.com/notebooks/basic_features_overview.ipynb)
* **Questions:** [Biostars](https://www.biostars.org/) and [Stack Overflow](https://stackoverflow.com/) for programming questions; the forums above for database-specific ones.

## Warm-up · The databases on the web (no code, 10 min)

Before writing any code, look at the same data the way most people do: on the websites. Keep these tabs open; you can compare them with what the code returns later.

▶ <font color="#C62828">**Your turn:**</font> open each link, do the search, and write down the answer.

**1. Open Targets Platform** – [platform.opentargets.org](https://platform.opentargets.org/)
* Search **semaglutide**. Under *Mechanisms of action*, which protein does it act on, and how (agonist or antagonist)?
* Search **GLP1R** (the target). In *Associated diseases*, which disease has the highest score?

**2. ChEMBL** – [ebi.ac.uk/chembl](https://www.ebi.ac.uk/chembl/)
* Search **sitagliptin**. What is its ChEMBL ID, and what is its highest clinical phase?
* On the sitagliptin page, find its **target** in *Drug Mechanisms*. How many bioactivity records does ChEMBL have for that target?

**3. Open Targets GraphQL browser** – [link](https://api.platform.opentargets.org/api/v4/graphql/browser)
* Explore the different queries can be made and the information field that can be requested for drugs, targets and diseases.

📊 <font color="#2E7D32">**What you get:**</font> you will see that the websites are easy for one search, but slow if you want 300 targets or 70 drugs. That is what the code below does for you.

## 0 · Setup (run once, about 1 minute)

▶ <font color="#C62828">**Your turn:**</font> run the four cells below, one after the other, with the ▶ button. You do not need to change anything.

🔎 <font color="#1565C0">**What this query does:**</font> installs two extra tools (the ChEMBL client to talk to ChEMBL, and RDKit to draw molecules), loads them, and defines a small helper `ot()` that sends questions to Open Targets.

In [ ]:
!pip install -q chembl_webresource_client rdkit

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated")   # hides a harmless warning

import requests
import pandas as pd
import matplotlib.pyplot as plt
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Draw
from google.colab import data_table
data_table.enable_dataframe_formatter()     # makes tables interactive: sort, filter, page through
pd.set_option("display.max_colwidth", 80)

In [ ]:
OT_URL = "https://api.platform.opentargets.org/api/v4/graphql"

def ot(query, variables=None):
    """Send a GraphQL query to Open Targets and return the 'data' part of the answer."""
    r = requests.post(OT_URL,
                      json={"query": query,
                            "variables": variables or {}},
                      timeout=120)
    r.raise_for_status()
    out = r.json()
    if out.get("errors"):
        raise RuntimeError(out["errors"])
    return out["data"]

def draw(df, smiles_col="smiles", label_col="chembl_id", n=20):
    """Draw the 2D structures of the first n molecules in a table."""
    sub = df.dropna(subset=[smiles_col]).head(n)
    mols = [Chem.MolFromSmiles(s) for s in sub[smiles_col]]
    return Draw.MolsToGridImage(mols,
                                molsPerRow=4,
                                subImgSize=(250, 200),
                                legends=[str(x) for x in sub[label_col]])

In [ ]:
meta = ot("{ meta { dataVersion { year month iteration } apiVersion { x y z } } }")["meta"]
print("Open Targets data version:", meta["dataVersion"])

📊 <font color="#2E7D32">**What you get:**</font> the Open Targets data release you are using (year and month).
Write it down: databases change, and results from a different release can differ slightly from your neighbour’s.

## A · Open Targets from Python

In the warm-up (and earlier in the workshop) you sent queries to Open Targets from the **GraphQL browser** and pressed ▶. Here, Python sends **exactly the same kind of text** to the same address. The only new idea is a **variable**: in the query you will see `$id`, a placeholder that Python fills in, so one query works for any drug, target or disease.

**The thread in Part A:** semaglutide (a diabetes and obesity drug) → its target GLP1R → the disease most strongly linked to GLP1R, type 2 diabetes.

### A1 · Find an identifier (name → ID)

Databases use IDs rather than names: ChEMBL IDs for drugs (e.g. CHEMBL2108724), Ensembl IDs for targets (ENSG…) and EFO/MONDO IDs for diseases.

🔎 <font color="#1565C0">**What this query does:**</font> the first cell defines a search question (`SEARCH_Q`) and a function `find_id(name, type)` that sends it. Nothing is printed yet.

In [ ]:
SEARCH_Q = """
query Search($q: String!, $types: [String!]) {
  search(
    queryString: $q,
    entityNames: $types,
    page: {index: 0, size: 5}) {
      hits {
        id
        name
        entity
        description }
  }
}
"""

def find_id(name, entity):
    """Search Open Targets for a name. entity is 'drug', 'target' or 'disease'."""
    hits = ot(SEARCH_Q, {"q": name,
                         "types": [entity]})["search"]["hits"]
    return pd.DataFrame(hits)[["id", "name", "entity", "description"]]

▶ <font color="#C62828">**Your turn:**</font> run the cell below. It searches for the drug **semaglutide**.

📊 <font color="#2E7D32">**What you get:**</font> a table with the best matches: the ID in the first column, then the name and a short description. The first row (CHEMBL2108724) is semaglutide; we use this ID in A2.

✏️ **First edit:** afterwards, replace `"semaglutide"` with another drug name you know (keep the quotation marks) and run the cell again.

In [ ]:
find_id("semaglutide", "drug")   # ✏️ change the name, keep the quotes

### A2 · A drug: semaglutide (ID from A1)

🔎 <font color="#1565C0">**What this query does:**</font> asks Open Targets for everything it knows about one drug: its name and type, its highest clinical stage, a description, its **mechanism of action** (which target it acts on, and how), and every **indication** (disease it has been tested or approved for), with the highest stage reached for each.

The code then arranges the answer as one table: **one row per indication**, with the drug information and mechanism repeated on every row so that you can sort and filter it.

▶ <font color="#C62828">**Your turn:**</font> run the cell. Nothing needs changing.

In [ ]:
DRUG_ID = "CHEMBL2108724"   # semaglutide, the ID found in A1

DRUG_Q = """
query Drug($id: String!) {
  drug(chemblId: $id) {
    id
    name
    drugType
    maximumClinicalStage
    description
    mechanismsOfAction {
      rows {
        mechanismOfAction
        actionType
        targets {
          id
          approvedSymbol
        }
      }
    }
    indications {
      count
      rows {
        maxClinicalStage
        disease {
          id
          name
        }
      }
    }
  }
}
"""

def drug_table(chembl_id):
    d = ot(DRUG_Q, {"id": chembl_id})["drug"]
    info = pd.DataFrame([
        {"chembl_id": d["id"],
         "drug": d["name"],
         "drug_type": d["drugType"],
         "drug_max_stage": d["maximumClinicalStage"],
         "description": d["description"]}])

    moa = pd.DataFrame([
        {"mechanism": m["mechanismOfAction"],
         "action": m["actionType"],
         "target": t["approvedSymbol"],
         "target_id": t["id"]}

        for m in (d["mechanismsOfAction"] or {}).get("rows", [])
        for t in (m["targets"] or [{"approvedSymbol": None, "id": None}])
    ] or [{"mechanism": None, "action": None, "target": None, "target_id": None}])

    ind = pd.DataFrame([
        {"disease_id": r["disease"]["id"] if r["disease"] else None,
         "disease": r["disease"]["name"] if r["disease"] else None,
         "indication_stage": r["maxClinicalStage"]}

        for r in (d["indications"] or {}).get("rows", [])
    ] or [{"disease_id": None,
           "disease": None,
           "indication_stage": None}])

    return info.merge(moa, how="cross").merge(ind, how="cross")

drug_df = drug_table(DRUG_ID)
print(drug_df["drug"].iloc[0], "|", drug_df["drug_type"].iloc[0], "| highest stage:", drug_df["drug_max_stage"].iloc[0])
print("Mechanism:", drug_df["mechanism"].iloc[0], "→ target", drug_df["target"].iloc[0])
print(drug_df["disease"].nunique(), "indications")
drug_df.drop(columns="description").sort_values(["indication_stage", "disease"])

📊 <font color="#2E7D32">**What you get:**</font> first a short summary (drug type, highest stage, mechanism and target), then the full table. Use the filter icon above the table to find, for example, all rows where `indication_stage` is APPROVAL. The `target` column (GLP1R) is the input for A3.

The chart below shows the same information visually: how many indications semaglutide has reached at each clinical stage.

In [ ]:
stage_counts = drug_df.drop_duplicates("disease")["indication_stage"].value_counts()
stage_counts.sort_values().plot.barh(color="#1565C0", figsize=(6, 3))
plt.xlabel("number of indications")
plt.title("Semaglutide: indications by clinical stage")
plt.show()

### A3 · A target: GLP1R (target from A2)

▶ <font color="#C62828">**Your turn:**</font> run the cell below. It searches for the target **GLP1R**.

📊 <font color="#2E7D32">**What you get:**</font> a table with the best matches: the ID in the first column, then the name and a short description.

✏️ **First edit:** afterwards, replace `"GLP1R"` with another target name you know (keep the quotation marks) and run the cell again.

In [ ]:
find_id("GLP1R", "target")   # ✏️ change the name, keep the quotes

🔎 <font color="#1565C0">**What this query does:**</font> takes the found target and asks Open Targets for its **class** (e.g. GPCR), **UniProt ID**, **tractability** (can it be drugged, and with which kind of molecule), **subcellular location**, the **drugs and clinical candidates** acting on it, and the **diseases** most strongly associated with it.

In [ ]:
TARGET_ID = "ENSG00000112164"

TARGET_Q = """
query Target($id: String!, $nDiseases: Int!) {
  target(ensemblId: $id) {
    id
    approvedSymbol
    approvedName
    targetClass {
      label
      level
    }
    proteinIds {
      id
      source
    }
    tractability {
      modality
      label
      value
    }
    subcellularLocations {
      location
      source
    }
    drugAndClinicalCandidates {
      count
      rows {
        maxClinicalStage
        drug {
          id
          name
          drugType
        }
      }
    }
    associatedDiseases(page: {index: 0, size: $nDiseases}) {
      count
      rows {
        score
        disease {
          id
          name
        }
      }
    }
  }
}
"""

MODALITY = {"SM": "small molecule",
            "AB": "antibody",
            "PR": "PROTAC",
            "OC": "other"}

t = ot(TARGET_Q, {"id": TARGET_ID,
                  "nDiseases": 10})["target"]

summary_df = pd.DataFrame([{
    "target": t["approvedSymbol"], "ensembl_id": t["id"], "name": t["approvedName"],
    "class": "; ".join(sorted({c["label"] for c in t["targetClass"]})),
    "uniprot": ", ".join(p["id"] for p in t["proteinIds"] if p["source"] == "uniprot_swissprot"),
    "locations": "; ".join(sorted({l["location"] for l in t["subcellularLocations"]})),
    "tractable_as": ", ".join(sorted({MODALITY.get(x["modality"], x["modality"]) for x in t["tractability"] if x["value"]})),
    "n_drugs_and_candidates": t["drugAndClinicalCandidates"]["count"],
    "n_associated_diseases": t["associatedDiseases"]["count"],
}])

tract_df = pd.DataFrame([{"modality": MODALITY.get(x["modality"], x["modality"]),
                          "evidence": x["label"]}
                         for x in t["tractability"] if x["value"]])

target_drugs_df = pd.DataFrame([
    {"chembl_id": r["drug"]["id"],
     "drug": r["drug"]["name"],
     "drug_type": r["drug"]["drugType"],
     "max_stage": r["maxClinicalStage"]}
    for r in t["drugAndClinicalCandidates"]["rows"] if r["drug"]]).drop_duplicates()

target_diseases_df = pd.DataFrame([
    {"disease_id": r["disease"]["id"],
     "disease": r["disease"]["name"],
     "score": round(r["score"], 3)}
    for r in t["associatedDiseases"]["rows"]])

display(summary_df)

📊 <font color="#2E7D32">**What you get:**</font> one row summarising GLP1R: its class, UniProt ID, where it sits in the cell, which kinds of drug can reach it, and how many drugs and diseases are linked to it. The next cells show the details behind that summary.

In [ ]:
print("Tractability: which kinds of molecule can reach the target, and the evidence for it")
display(tract_df)

In [ ]:
print("Drugs and clinical candidates acting on the target")
display(target_drugs_df)

In [ ]:
print("Associated diseases of the target")
display(target_diseases_df)

In [ ]:
target_diseases_df.sort_values("score").plot.barh(x="disease", y="score", color="#2E7D32", legend=False, figsize=(7, 4))
plt.xlabel("association score (0–1)")
plt.ylabel("")
plt.title(f"Diseases most associated with {t['approvedSymbol']}")
plt.show()

📊 <font color="#2E7D32">**What you get:**</font> the ten diseases most strongly associated with GLP1R. The longest bar is type 2 diabetes: that is the input for A4.

### A4 · A disease: type 2 diabetes (top disease from A3)

🔎 <font color="#1565C0">**What this query does:**</font> takes the disease most associated with GLP1R in A3 and asks for its description, therapeutic areas, all **drugs and clinical candidates** for it, and the **targets** most strongly associated with it.

▶ <font color="#C62828">**Your turn:**</font> run the cell. Check that the printed disease is *type 2 diabetes mellitus* (ID MONDO_0005148); if not, type that ID in the first line instead.

In [ ]:
DISEASE_ID = target_diseases_df.sort_values("score", ascending=False)["disease_id"].iloc[0]   # from A3
print("Disease ID from A3:", DISEASE_ID)

DISEASE_Q = """
query Disease($id: String!) {
  disease(efoId: $id) {
    name
    description
    therapeuticAreas {
      name
    }
    drugAndClinicalCandidates {
      count
      rows {
        maxClinicalStage
        drug {
          id
          name
          drugType
        }
      }
    }
    associatedTargets(page: {index: 0, size: 10}) {
      count
      rows {
        score
        target {
          id
          approvedSymbol
          approvedName
        }
      }
    }
  }
}
"""

dis = ot(DISEASE_Q, {"id": DISEASE_ID})["disease"]
print(dis["name"], "|", [a["name"] for a in dis["therapeuticAreas"]])
print(dis["description"])
print("\nDrugs and clinical candidates:", dis["drugAndClinicalCandidates"]["count"])
print("Associated targets:", dis["associatedTargets"]["count"])

disease_drugs_df = pd.DataFrame([{"chembl_id": r["drug"]["id"],
                                  "drug": r["drug"]["name"],
                                  "drug_type": r["drug"]["drugType"],
                                  "max_stage": r["maxClinicalStage"]}
                                 for r in dis["drugAndClinicalCandidates"]["rows"] if r["drug"]]).drop_duplicates()

disease_targets_df = pd.DataFrame([{"target": r["target"]["approvedSymbol"],
                                    "name": r["target"]["approvedName"],
                                    "score": round(r["score"], 3)} for r in dis["associatedTargets"]["rows"]])

display(disease_targets_df)

📊 <font color="#2E7D32">**What you get:**</font> the disease description, how many drugs and targets are linked to it (note how many more targets than drugs: most associated targets have never been drugged), and the 10 most associated targets. The chart below shows the drugs for the disease by clinical stage.

In [ ]:
disease_drugs_df["max_stage"].value_counts().sort_values().plot.barh(color="#6A1B9A", figsize=(6, 4))
plt.xlabel("number of drugs and candidates")
plt.title(f"{dis['name']}: drugs by clinical stage")
plt.show()

print("Approved drugs for this disease (first 20):")
display(disease_drugs_df[disease_drugs_df["max_stage"] == "APPROVAL"].head(20))

📊 <font color="#2E7D32">**What you get:**</font> most candidates never reach approval. In the list of approved drugs you will find **SITAGLIPTIN**: we follow it into ChEMBL in Part B.

## B · ChEMBL with Python

Open Targets tells us **which** targets and drugs matter; ChEMBL tells us **which molecules** have been measured on a target and **how potent** they are.

Here we use the `chembl_webresource_client` package, which turns each ChEMBL question into one line of Python. Each line has the form `new_client.<resource>.filter(<conditions>)`, for example `new_client.molecule.filter(pref_name__iexact="sitagliptin")`. The full list of resources (molecule, target, activity, similarity, …) and the conditions you can use is in the [ChEMBL API documentation](https://www.ebi.ac.uk/chembl/api/data/docs); more examples are in the [client’s GitHub page](https://github.com/chembl/chembl_webresource_client).

**The thread in Part B:** sitagliptin (an approved diabetes drug from A4) → its target DPP4 → the most potent compounds measured on DPP4 → molecules similar to sitagliptin.

### B1 · Look up a molecule (drug from A4)

🔎 <font color="#1565C0">**What this query does:**</font> this query looks up the molecule **sitagliptin** in ChEMBL by name and returns its ChEMBL ID, highest clinical phase, molecule type and structure (as SMILES and InChIKey, from the lecture).

▶ <font color="#C62828">**Your turn:**</font> run the cell.

In [ ]:
DRUG_NAME = "sitagliptin"   # ✏️ try "imatinib" or "ibuprofen" afterwards

mols = new_client.molecule.filter(pref_name__iexact=DRUG_NAME).only(
    ["molecule_chembl_id", "pref_name", "max_phase", "molecule_type", "molecule_structures"])
m = mols[0]
print(m["molecule_chembl_id"], m["pref_name"], "| max phase:", m["max_phase"], "|", m["molecule_type"])
print("SMILES:  ", m["molecule_structures"]["canonical_smiles"])
print("InChIKey:", m["molecule_structures"]["standard_inchi_key"])
Draw.MolToImage(Chem.MolFromSmiles(m["molecule_structures"]["canonical_smiles"]), size=(350, 250))

📊 <font color="#2E7D32">**What you get:**</font> sitagliptin’s ChEMBL ID, its maximum phase (4 = approved), its SMILES and InChIKey, and a drawing of its structure. The SMILES is the input for B4.

✏️ Afterwards, change `"sitagliptin"` to `"imatinib"` or `"ibuprofen"` and run it again; then change it back to `"sitagliptin"` before continuing.

### B2 · From a drug to its target in ChEMBL

🔎 <font color="#1565C0">**What this query does:**</font> uses the A2 function to ask Open Targets for sitagliptin’s mechanism of action (its target is DPP4), then translates the target to ChEMBL: from the Ensembl ID to the UniProt accession (Open Targets), and from UniProt to the ChEMBL target ID (ChEMBL). This is the kind of identifier mapping discussed in the lecture.

▶ <font color="#C62828">**Your turn:**</font> run the cell.

In [ ]:
UNIPROT_Q = """
query T($id: String!) {
  target(ensemblId: $id) {
    approvedSymbol
    proteinIds {
      id
      source
    }
  }
}
"""

def ensembl_for_symbol(symbol):
    hits = find_id(symbol, "target")
    exact = hits[hits["name"].str.upper() == symbol.upper()]
    return (exact if len(exact) else hits).iloc[0]["id"]

def chembl_target_for_ensembl(ensembl_id):
    t = ot(UNIPROT_Q, {"id": ensembl_id})["target"]
    accs = [p["id"] for p in t["proteinIds"] if p["source"] == "uniprot_swissprot"] or \
           [p["id"] for p in t["proteinIds"] if "uniprot" in p["source"].lower()]

    for acc in accs:
        hits = new_client.target.filter(target_components__accession=acc,
                                        target_type="SINGLE PROTEIN").only(
                                        ["target_chembl_id", "pref_name", "organism"])
        hits = [h for h in hits if h["organism"] == "Homo sapiens"]
        if hits:
            return hits[0]["target_chembl_id"], hits[0]["pref_name"], acc
    return None, None, (accs[0] if accs else None)

sita = drug_table(m["molecule_chembl_id"])            # Open Targets, using the ChEMBL ID from B1
target_symbol = sita["target"].dropna().iloc[0]
target_ensembl = sita["target_id"].dropna().iloc[0]
chembl_tid, chembl_tname, uniprot = chembl_target_for_ensembl(target_ensembl)
print(f"{m['pref_name']} acts on {target_symbol}: Ensembl {target_ensembl} → UniProt {uniprot} → ChEMBL {chembl_tid} ({chembl_tname})")

📊 <font color="#2E7D32">**What you get:**</font> one line showing the chain of IDs for sitagliptin’s target, DPP4. The ChEMBL target ID is the input for B3.

### B3 · Potent compounds on the target (target from B2)

🔎 <font color="#1565C0">**What this query does:**</font> asks ChEMBL for all measured activities on DPP4 with a potency of pChEMBL ≥ 7 (100 nM or better), keeping only exact values (“=”) and the activity types IC50, Ki, Kd and EC50. The code then combines repeated measurements into **one median value per compound** (the cleaning steps from the lecture).

▶ <font color="#C62828">**Your turn:**</font> run the cell. It downloads up to 500 records and can take 20–30 seconds.

In [ ]:
def potent_compounds(target_chembl_id, min_pchembl=7, max_records=500):
    acts = new_client.activity.filter(
        target_chembl_id=target_chembl_id,
        pchembl_value__gte=min_pchembl,
        standard_relation="=",
        standard_type__in=["IC50", "Ki", "Kd", "EC50"],
    ).only(["molecule_chembl_id", "canonical_smiles", "standard_type", "pchembl_value"])

    rows = []
    for i, a in enumerate(acts):
        rows.append(a)
        if i + 1 >= max_records:
            break
    df = pd.DataFrame(rows)

    if df.empty:
        return df
    df["pchembl_value"] = df["pchembl_value"].astype(float)

    return (df.groupby(["molecule_chembl_id", "canonical_smiles"])
              .agg(pchembl=("pchembl_value", "median"), n=("pchembl_value", "size"))
              .reset_index().sort_values("pchembl", ascending=False))

cmpds = potent_compounds(chembl_tid)
print(len(cmpds), f"potent compounds on {chembl_tname} (from the first 500 records)")
cmpds.head(10)

📊 <font color="#2E7D32">**What you get:**</font> a table of compounds ranked by potency (`pchembl`: higher is more potent; 8 = 10 nM) and `n`, the number of measurements behind each median. The chart and drawings below show the distribution and the most potent structures.

In [ ]:
cmpds["pchembl"].plot.hist(bins=20, color="#1565C0", figsize=(6, 3))
plt.xlabel("pChEMBL (higher = more potent)")
plt.ylabel("number of compounds")
plt.title(f"Potency of compounds measured on {chembl_tname}")
plt.show()


cmpds["label"] = cmpds["molecule_chembl_id"] + "  " + cmpds["pchembl"].astype('str')
draw(cmpds, smiles_col="canonical_smiles", label_col="label", n=8)

✏️ Optional: to repeat this for the dopamine D2 receptor from the lecture, run `potent_compounds(chembl_target_for_ensembl(ensembl_for_symbol("DRD2"))[0])` in a new cell.

### B4 · Molecules similar to sitagliptin (SMILES from B1)

🔎 <font color="#1565C0">**What this query does:**</font> sends sitagliptin’s structure (SMILES) to ChEMBL’s similarity search, which returns every molecule whose structure is at least 70% similar.

▶ <font color="#C62828">**Your turn:**</font> run the two cells below.

In [ ]:
def similar_molecules(smiles, similarity=70, max_records=100):
    res = new_client.similarity.filter(smiles=smiles, similarity=similarity).only(
        ["molecule_chembl_id", "pref_name", "similarity", "max_phase", "molecule_structures"])
    rows = []
    for i, r in enumerate(res):
        s = r.get("molecule_structures") or {}
        rows.append({"chembl_id": r["molecule_chembl_id"],
                     "name": r["pref_name"],
                     "similarity": float(r["similarity"]),
                     "max_phase": r["max_phase"],
                     "smiles": s.get("canonical_smiles")})

        if i + 1 >= max_records:
            break
    return pd.DataFrame(rows)

query_smiles = m["molecule_structures"]["canonical_smiles"]   # from B1
SIMILARITY = 70   # ✏️ try 85 (only very close analogues) or 60 (further away)
sim = similar_molecules(query_smiles, similarity=SIMILARITY)
print(len(sim), f"molecules at least {SIMILARITY}% similar to {m['pref_name']}")
sim.head(15)

In [ ]:
sim["similarity"].plot.hist(bins=15, color="#2E7D32", figsize=(6, 3))
plt.xlabel("similarity to the query (%)")
plt.ylabel("number of molecules")
plt.title("How similar are the hits?")
plt.show()

sim["label"] = sim["chembl_id"] + "  " + sim["similarity"].round(0).astype(int).astype(str) + "%"
draw(sim, label_col="label", n=16)

📊 <font color="#2E7D32">**What you get:**</font> a table of similar molecules (with any name and clinical phase), a histogram of how similar they are, and their 2D structures labelled with ChEMBL ID and similarity. The first hits are sitagliptin itself (100%, including its salt forms); the rest share its core. Compare which parts of the structure change between the analogues.

## C · Worked example: a type 2 diabetes drug with a **novel** mechanism of action

Now we put the pieces together for a real question, using the disease from A4.

**Question:** which proteins could be targeted to treat type 2 diabetes, that are **not** already the targets of approved diabetes drugs?

| Step | Database | What we do | Output used in |
|---|---|---|---|
| 1 | Open Targets | All targets associated with type 2 diabetes, with their evidence | Step 3 |
| 2 | Open Targets (from ChEMBL) | All drugs for type 2 diabetes, their mechanism of action and targets | Step 3 |
| 3 | – | Remove the targets of approved drugs: what is left is "novel" | Step 4 |
| 4 | Open Targets | Profile the candidates: class, tractability, location, drugs for other diseases | Step 5 |
| 5 | ChEMBL | For one candidate: known potent compounds and similar molecules | your conclusions |

In Part D you will run exactly these steps for your own disease, by changing the ID in the next cell.

▶ <font color="#C62828">**Your turn:**</font> run the cell below. It uses the disease from A4 (type 2 diabetes).

In [ ]:
find_id("diabetes", "disease")   # ✏️ change the name, keep the quotes

In [ ]:
C_DISEASE_ID = DISEASE_ID   # ✏️ in Part D, replace with your disease ID, e.g. "MONDO_0011122"
N_TARGETS = 300             # how many top-associated targets to download
print("Disease for Part C:", C_DISEASE_ID)

### Step 1 · Targets associated with the disease

🔎 <font color="#1565C0">**What this query does:**</font> downloads the 300 targets most associated with the disease, with the overall score (0–1) and a score for each type of evidence (genetic association, literature, animal models, known drugs, …).

▶ <font color="#C62828">**Your turn:**</font> run the cell (about 10 seconds).

In [ ]:
ASSOC_Q = """
query Assoc($id: String!, $index: Int!, $size: Int!) {
  disease(efoId: $id) {
    name
    associatedTargets(page: {index: $index, size: $size}) {
      count
      rows {
        score
        datatypeScores {
          id
          score
        }
        target {
          id
          approvedSymbol
          approvedName
        }
      }
    }
  }
}
"""

def associated_targets(disease_id, n=300, page_size=100):
    rows, name = [], None
    for index in range((n + page_size - 1) // page_size):
        d = ot(ASSOC_Q, {"id": disease_id, "index": index, "size": page_size})["disease"]
        name = d["name"]
        for r in d["associatedTargets"]["rows"]:
            row = {"ensembl": r["target"]["id"],
                   "symbol": r["target"]["approvedSymbol"],
                   "name": r["target"]["approvedName"],
                   "score": r["score"]}
            row.update({dt["id"]: dt["score"] for dt in r["datatypeScores"]})
            rows.append(row)

        if len(d["associatedTargets"]["rows"]) < page_size:
            break
    df = pd.DataFrame(rows).fillna(0)
    print(f"{name}: {d['associatedTargets']['count']} associated targets in total, {len(df)} downloaded")
    return df, name

assoc, disease_name = associated_targets(C_DISEASE_ID, N_TARGETS)
assoc.head(15).round(3)

📊 <font color="#2E7D32">**What you get:**</font> one row per target, with its overall score and one column per evidence type. The heat map below shows the same for the top 20 targets: each column is an evidence type, darker means stronger. Notice which targets are supported by **genetics** and which mostly by **literature**.

In [ ]:
ev_cols = [c for c in ["genetic_association", "somatic_mutation", "known_drug", "clinical", "affected_pathway",
                       "rna_expression", "animal_model", "literature"] if c in assoc.columns]
top20 = assoc.head(20).set_index("symbol")[ev_cols]
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(top20.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(ev_cols)))
ax.set_xticklabels(ev_cols, rotation=45, ha="right")
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index)
plt.colorbar(im, label="evidence score")
ax.set_title(f"{disease_name}: evidence for the top 20 targets")
plt.tight_layout()
plt.show()

### Step 2 · Drugs already used or tested for the disease, and their targets

🔎 <font color="#1565C0">**What this query does:**</font> asks for every drug and clinical candidate for the disease, with its highest clinical stage and its mechanism of action, and lists one row per drug and target.

▶ <font color="#C62828">**Your turn:**</font> run the cell.

In [ ]:
DRUGS_Q = """
query DiseaseDrugs($id: String!) {
  disease(efoId: $id) {
    drugAndClinicalCandidates {
      count
      rows {
        maxClinicalStage
        drug {
          id
          name
          drugType
          mechanismsOfAction {
            rows {
              mechanismOfAction
              actionType
              targets {
                id
                approvedSymbol
              }
            }
          }
        }
      }
    }
  }
}
"""

def disease_drugs(disease_id):
    rows = []
    for r in ot(DRUGS_Q, {"id": disease_id})["disease"]["drugAndClinicalCandidates"]["rows"]:
        d = r["drug"]
        if d is None:
            continue
        moas = d["mechanismsOfAction"]["rows"] if d["mechanismsOfAction"] else []
        targets = [(t["id"], t["approvedSymbol"], m["mechanismOfAction"], m["actionType"])
                   for m in moas for t in m["targets"]] or [(None, None, None, None)]
        for ens, sym, mech, act in targets:
            rows.append({"drug": d["name"],
                         "chembl_id": d["id"],
                         "type": d["drugType"],
                         "stage": r["maxClinicalStage"],
                         "target_ensembl": ens,
                         "target": sym,
                         "mechanism": mech,
                         "action": act})

    return pd.DataFrame(rows).drop_duplicates()

def is_approved(stage):
    s = str(stage).upper()
    return "APPROV" in s or "PHASE_4" in s or s == "4"

drugs = disease_drugs(C_DISEASE_ID)
drugs["approved"] = drugs["stage"].map(is_approved)
approved = drugs[drugs["approved"]]
print("Clinical stages found:", drugs["stage"].value_counts().to_dict())
print(approved["drug"].nunique(), "approved drugs for", disease_name)

moa_summary = (approved.dropna(subset=["target"])
               .groupby(["target", "action"])
               .agg(drugs=("drug", lambda x: ", ".join(sorted(set(x))[:6])), n_drugs=("drug", "nunique"))
               .reset_index().sort_values("n_drugs", ascending=False))
moa_summary

📊 <font color="#2E7D32">**What you get:**</font> the clinical stages found, the number of approved drugs, and a table of the targets of approved drugs, with how many drugs act on each. The chart shows the 15 most used targets: the known mechanisms for this disease.

In [ ]:
moa_summary.groupby("target")["n_drugs"].sum().sort_values().tail(15).plot.barh(color="#6A1B9A", figsize=(6, 5))
plt.xlabel("approved drugs acting on the target")
plt.title(f"{disease_name}: targets of approved drugs")
plt.show()

### Step 3 · Remove targets that approved drugs already hit (inputs from Steps 1 and 2)

🔎 <font color="#1565C0">**What this query does:**</font> no new query: we take the targets from Step 1 and remove every target that an approved drug from Step 2 already acts on. Targets that are only in clinical trials for the disease stay, but are flagged. The remaining list is sorted by genetic evidence first.

▶ <font color="#C62828">**Your turn:**</font> run the cell.

In [ ]:
approved_targets = set(approved["target_ensembl"].dropna())
trial_targets = set(drugs.loc[~drugs["approved"], "target_ensembl"].dropna()) - approved_targets

candidates = assoc[~assoc["ensembl"].isin(approved_targets)].copy()
candidates["in_trials_for_disease"] = candidates["ensembl"].isin(trial_targets)
print(len(approved_targets), "targets of approved drugs removed")
print(candidates["in_trials_for_disease"].sum(), "remaining targets are already in clinical trials for this disease")

if "genetic_association" in candidates.columns:
    candidates = candidates.sort_values(["genetic_association", "score"], ascending=False)

cols = [c for c in ["symbol", "name", "score", "genetic_association", "literature",
                    "animal_model", "in_trials_for_disease"] if c in candidates.columns]
candidates[cols].head(20).round(3)

📊 <font color="#2E7D32">**What you get:**</font> the candidate targets that no approved drug for this disease acts on yet, with the strongest genetic evidence at the top. These are the input for Step 4.

### Step 4 · Profile the top candidates (candidates from Step 3)

🔎 <font color="#1565C0">**What this query does:**</font> one query for the top 15 candidates at once: target class, tractability by modality, subcellular location, and the drugs acting on each target for **any** disease.

▶ <font color="#C62828">**Your turn:**</font> run the cell. ✏️ You can change `TOP_N` to profile more or fewer candidates.

In [ ]:
PROFILE_Q = """
query Profile($ids: [String!]!) {
  targets(ensemblIds: $ids) {
    id approvedSymbol
    targetClass {
      label
      level
    }
    tractability {
      modality
      label
      value
    }
    subcellularLocations {
      location
    }
    drugAndClinicalCandidates {
      count
      rows {
        maxClinicalStage
        drug {
          name
          drugType
        }
      }
    }
  }
}
"""

def profile_targets(ensembl_ids):
    out = []
    for t in ot(PROFILE_Q, {"ids": list(ensembl_ids)})["targets"]:
        tract = {}
        for x in t["tractability"]:
            if x["value"]:
                tract.setdefault(MODALITY.get(x["modality"], x["modality"]), []).append(x["label"])
        rows = t["drugAndClinicalCandidates"]["rows"]
        approved_any = sorted({r["drug"]["name"] for r in rows if r["drug"] and is_approved(r["maxClinicalStage"])})
        out.append({
            "symbol": t["approvedSymbol"],
            "class": "; ".join(sorted({c["label"] for c in t["targetClass"]}))[:60],
            "tractable_as": ", ".join(tract) or "none reported",
            "locations": "; ".join(sorted({l["location"] for l in t["subcellularLocations"]})[:4]),
            "compounds_any_disease": len({r["drug"]["name"] for r in rows if r["drug"]}),
            "compounds_approved_any_disease": ", ".join(approved_any[:5]),
            "compounds_types": ", ".join(sorted({r["drug"]["drugType"] for r in rows if r["drug"]})),
        })
    return pd.DataFrame(out)

TOP_N = 15   # ✏️ how many candidates to profile
top = candidates.head(TOP_N)
profile = top[["symbol", "score"] + [c for c in ["genetic_association"] if c in top.columns]].merge(
    profile_targets(top["ensembl"]), on="symbol")
profile.round(3)

📊 <font color="#2E7D32">**What you get:**</font> one row per candidate. How to read it:
* *tractable_as* = “none reported” → hard to drug with known modalities; a small-molecule pocket, or a secreted or membrane protein (antibody), is a good sign.
* *compounds_approved_any_disease* not empty → a **repurposing** opportunity: a drug exists, but not for this disease.
* *locations*: plasma membrane or extracellular → reachable by antibodies and peptides; intracellular → small molecules.

The chart below places each candidate by its overall score and genetic evidence. Targets in the **top right** have both; filled circles can be reached by a small molecule.

In [ ]:
sm = profile["tractable_as"].str.contains("small molecule")
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(profile.loc[sm, "score"],
           profile.loc[sm, "genetic_association"],
           s=80,
           color="#1565C0",
           label="small-molecule tractable")

ax.scatter(profile.loc[~sm, "score"],
           profile.loc[~sm, "genetic_association"],
           s=80,
           facecolors="none",
           edgecolors="#1565C0",
           label="not small-molecule tractable")

for _, r in profile.iterrows():
    ax.annotate(r["symbol"],
                (r["score"], r["genetic_association"]),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=9)

ax.set_xlabel("overall association score")
ax.set_ylabel("genetic association score")
ax.set_title(f"{disease_name}: candidate targets")
ax.legend()
plt.show()

**Expression.**

🔎 <font color="#1565C0">**What this query does:**</font> the cell below asks Open Targets in which tissues a target is most expressed.

▶ <font color="#C62828">**Your turn:**</font> ✏️ type a candidate’s symbol (default: the first one in the table) and run.

📊 <font color="#2E7D32">**What you get:**</font> the tissues with the highest measured expression; you can also check the target in the [Human Protein Atlas](https://www.proteinatlas.org/).

In [ ]:
EXPR_Q = """
query Expr($id: String!) {
  target(ensemblId: $id) {
    approvedSymbol
    baselineExpression(page: {index: 0, size: 200}) {
      rows {
        datasourceId
        unit
        median
        tissueBiosample {
          biosampleName
        }
      }
    }
  }
}
"""

def top_tissues(symbol, n=10):
    ens = candidates.loc[candidates["symbol"] == symbol, "ensembl"]
    ens = ens.iloc[0] if len(ens) else ensembl_for_symbol(symbol)
    rows = ot(EXPR_Q, {"id": ens})["target"]["baselineExpression"]["rows"]
    df = pd.DataFrame([{"tissue": (r["tissueBiosample"] or {}).get("biosampleName"),
                        "source": r["datasourceId"],
                        "median": r["median"],
                        "unit": r["unit"]} for r in rows]).dropna(subset=["tissue", "median"])
    return df.sort_values("median", ascending=False).head(n)

EXPR_SYMBOL = profile.iloc[0]["symbol"]   # ✏️ or e.g. "GCK"
tt = top_tissues(EXPR_SYMBOL)
tt.sort_values("median").plot.barh(x="tissue", y="median", color="#2E7D32", legend=False, figsize=(6, 4))
plt.xlabel(f"median expression ({tt['unit'].iloc[0]})" if len(tt) else "")
plt.ylabel("")
plt.title(f"{EXPR_SYMBOL}: highest-expressing tissues")
plt.show()

### Step 5 · Chemistry for one candidate (candidate from Step 4)

🔎 <font color="#1565C0">**What this query does:**</font> maps your chosen candidate to its ChEMBL target (as in B2), retrieves its most potent compounds (as in B3, pChEMBL ≥ 6), and finds molecules similar to the most potent one (as in B4).

▶ <font color="#C62828">**Your turn:**</font> ✏️ choose a candidate from the Step 4 table that is tractable as a small molecule, type its symbol in the first line, and run the two cells.

In [ ]:
CHOSEN = profile.iloc[0]["symbol"]   # ✏️ replace with your chosen symbol, e.g. "GCK"

ens = candidates.loc[candidates["symbol"] == CHOSEN, "ensembl"].iloc[0]
tid, tname, acc = chembl_target_for_ensembl(ens)
print(CHOSEN, "→ UniProt", acc, "→ ChEMBL", tid, tname)

cmpds_c = potent_compounds(tid, min_pchembl=6) if tid else pd.DataFrame()
print(len(cmpds_c), "compounds with pChEMBL ≥ 6")
cmpds_c.head(10)

In [ ]:
if len(cmpds_c):
    best = cmpds_c.iloc[0]   # ✏️ use .iloc[1] for the second most potent compound, and so on
    print("Most potent compound:", best["molecule_chembl_id"], "pChEMBL", best["pchembl"])
    sim_c = similar_molecules(best["canonical_smiles"], similarity=50)
    display(sim_c.head(15))
    sim_c["label"] = sim_c["chembl_id"] + "  " + sim_c["similarity"].round(0).astype(int).astype(str) + "%"
    display(draw(sim_c, label_col="label", n=12))
else:
    print("No ChEMBL bioactivity data: an opportunity for new chemistry, or a target better suited to biologics.")

📊 <font color="#2E7D32">**What you get:**</font> the ChEMBL target for your candidate, its most potent compounds, and a series of similar molecules with their structures. If there are none, that is a result too: the target is chemically unexplored.

### Discussion
* Which candidates have strong *genetic* evidence and which rely mostly on literature?
* Which ones already have approved drugs for other diseases (repurposing), and which have none?
* Would you go for a small molecule, an antibody or another modality, given location and tractability?
* What could go wrong: safety, expression in other tissues, redundancy with existing mechanisms?

## D · Your group project

You now repeat Part C for a disease of your choice. Nothing new to learn: you only change the disease ID.

▶ <font color="#C62828">**Your turn:**</font>
1. **Find your disease ID.** ✏️ Change the name in the cell below and run it. Copy the ID from the first column (for example MONDO_0011122).
2. **Set it in Part C.** Scroll up to the first cell of Part C and replace `DISEASE_ID` with your ID in quotes, e.g. `C_DISEASE_ID = "MONDO_0011122"`.
3. **Rerun Part C.** Click that cell, then use the menu **Runtime ▸ Run after** to run every cell below it.
4. **Choose 3 candidate targets** and, for each, note: evidence (score, genetic association), drugs already acting on it (any disease), best modality, subcellular location, expression, and an example compound series from ChEMBL if one exists.
5. **Prepare a 5-minute presentation** (3–4 slides).

Some ideas for diseases:
* Obesity, beyond the incretin (GLP-1, GIP) receptors
* Alzheimer’s disease, beyond amyloid-β, acetylcholinesterase and NMDA receptors
* Asthma, deciding for each target whether an inhaled small molecule or an injected antibody would fit best
* Rheumatoid arthritis, beyond the TNF, IL-6 and JAK inhibitors
* Schizophrenia, beyond the dopamine D2 and serotonin 5-HT2A receptors
* A disease from your own research, with the same question

In [ ]:
find_id("obesity", "disease")   # ✏️ e.g. "Alzheimer disease", "asthma", "rheumatoid arthritis", "schizophrenia"